# 05 · Clustering & pattern discovery
Cluster the latent space, select k by silhouette, and interpret clusters against conventional faults.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dga.config import load_config, set_seed
cfg = load_config(); set_seed(cfg.seed)

In [ ]:
import numpy as np
from dga.config import PROJECT_ROOT
Z = np.load(PROJECT_ROOT/'results'/'models'/'latent.npy')
index = np.load(PROJECT_ROOT/'results'/'models'/'feature_index.npy', allow_pickle=True)
from dga import clustering
res = clustering.fit(Z, cfg.clustering)
print(res.algorithm, 'k=', res.k, 'silhouette=', round(res.silhouette,3))

## Silhouette vs k

In [ ]:
s = res.extra.get('silhouette_by_k', {})
fig, ax = plt.subplots(figsize=(4,3)); ax.plot(list(s), list(s.values()), 'o-')
ax.set_xlabel('k'); ax.set_ylabel('silhouette'); fig

## Cluster x fault profile

In [ ]:
from dga import data as dga_data, conventional, evaluation
df = dga_data.load_clean().loc[index]
duval = conventional.diagnose(df)['duval'].to_numpy()
evaluation.cluster_fault_profile(res.labels, duval)

In [ ]:
print('internal:', evaluation.internal_metrics(Z, res.labels))
print('external vs Duval:', evaluation.external_metrics(res.labels, duval))